# Phase 10 — L-CiteEval Pilot: Spectral Features for Grounded Generation

**Benchmark**: L-CiteEval / HotpotQA subtask (multi-doc QA, 8K–48K context)
**Model**: Falcon-3-10B-Instruct (T=1.0)
**Goal**: Determine if spectral features of $H(n)$ predict statement-level grounding faithfulness on long-context document QA.

---

In [ ]:
import os, sys, shutil

REPO_DIR = '/content/hallucination_detection'

# Remove stale clone if spectral_utils is missing
if os.path.exists(REPO_DIR) and not os.path.exists(os.path.join(REPO_DIR, 'spectral_utils')):
    shutil.rmtree(REPO_DIR)

if not os.path.exists(REPO_DIR):
    os.system(f'git clone -b gemini/phase10-pilot https://github.com/omrisegev/hallucination_detection.git {REPO_DIR}')
else:
    os.system(f'git -C {REPO_DIR} pull -q')

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Fix for NumPy 2.0 / SciPy conflict in Colab
os.system('pip install -q "numpy<2.0" "transformers>=4.40" accelerate datasets bitsandbytes autoawq gptqmodel scipy')

from spectral_utils import (
    load_model, generate_full, free_memory,
    extract_all_features, sw_var_peak_with_window, sw_var_peak_adaptive,
    segment_by_citations,
    load_lciteeval, lciteeval_prompt,
    FEAT_NAMES, load_cache, save_cache,
    zscore, boot_auc, nadler_fuse, simple_average_fusion, best_nadler_on,
)
print('spectral_utils imported OK')

In [ ]:
import torch
import numpy as np
from tqdm.auto import tqdm
import pickle

MODEL_ID = 'tiiuae/Falcon3-10B-Instruct'
TEMP = 1.0
MAX_NEW = 1024
N_SAMPLES = 100
TASK = 'hotpotqa'
CACHE_DIR = '/content/drive/MyDrive/hallucination_detection/cache/phase10_pilot'
EXP_NAME = f'falcon3_10b_{TASK}_pilot'

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

os.makedirs(CACHE_DIR, exist_ok=True)

In [ ]:
mdl, tok = load_model(MODEL_ID, quantize_4bit=False)

In [ ]:
data = load_lciteeval(TASK, n_samples=N_SAMPLES)

In [ ]:
results = []
checkpoint_path = os.path.join(CACHE_DIR, f"{EXP_NAME}_raw.pkl")

if os.path.exists(checkpoint_path):
    with open(checkpoint_path, 'rb') as f:
        results = pickle.load(f)
    print(f"Loaded {len(results)} samples from checkpoint.")

start_idx = len(results)

for i in tqdm(range(start_idx, len(data))):
    row = data[i]
    prompt = lciteeval_prompt(row)
    
    # generate_full returns (tokens, token_logprobs, token_entropies, full_text, token_offsets)
    res = generate_full(mdl, tok, prompt, T=TEMP, max_new=MAX_NEW)
    
    results.append({
        'idx': i,
        'gold': row,
        'output': res
    })
    
    if (i + 1) % 25 == 0:
        with open(checkpoint_path, 'wb') as f:
            pickle.dump(results, f)

with open(checkpoint_path, 'wb') as f:
    pickle.dump(results, f)
print("Inference complete.")

In [ ]:
del mdl, tok
free_memory()

In [ ]:
statement_data = []

for res in results:
    gen_text = res['output']['full_text']
    offsets = res['output']['token_offsets']
    
    segments = segment_by_citations(gen_text, offsets)
    
    for seg in segments:
        seg['sample_idx'] = res['idx']
        seg['ents'] = res['output']['token_entropies'][seg['token_start']:seg['token_end']]
        statement_data.append(seg)

print(f"Segmented into {len(statement_data)} statements.")

In [ ]:
valid_statements = []
all_feats = []

for seg in statement_data:
    ents = seg['ents']
    if len(ents) < 10: # FFT minimum
        continue
    
    feats = extract_all_features(ents)
    if feats:
        # Add adaptive variance peak
        feats['sw_var_peak_adaptive'] = sw_var_peak_adaptive(ents)
        
        valid_statements.append(seg)
        all_feats.append(feats)

print(f"Retained {len(valid_statements)} valid statements (len >= 10).")

### Cell 10 — Ground Truth Labels

We use L-CiteEval's Citation Recall (CR) logic. A statement is grounded (1) if its cited sentence supports it. Since we don't have the full L-CiteEval scorer local, we'll implement a simplified version or use the dataset's provided `citations` list if available.

In [ ]:
def is_statement_grounded(seg, gold_row):
    """
    Simplified groundedness labeler.
    In a real run, this would call the L-CiteEval / ALCE scorer logic.
    For the pilot, we check if the citation index exists in the gold supporting facts.
    """
    # Extract supporting fact titles/indices from gold
    # L-CiteEval schema: 'supporting_facts' often contains (title, sent_idx)
    gold_cites = gold_row.get('supporting_facts', [])
    # For simplicity, if any cited ID is in gold, mark as grounded
    # This is a proxy for the pilot.
    return 1 if any(cid in [x[1] for x in gold_cites] for cid in seg['citation_ids']) else 0

labels = [is_statement_grounded(s, data[s['sample_idx']]) for s in valid_statements]
print(f"Class balance: {sum(labels)} Grounded / {len(labels) - sum(labels)} Ungrounded")

In [ ]:
import pandas as pd
from sklearn.metrics import roc_auc_score

feat_keys = list(all_feats[0].keys())
auc_results = []

for k in feat_keys:
    scores = np.array([f[k] for f in all_feats])
    # Sign orientation: most features we expect higher -> incorrect (hallucination)
    # But Nadler expects higher -> correct. We'll handle this in the AUC calculation.
    auc = roc_auc_score(labels, -scores)
    if auc < 0.5: auc = 1 - auc
    
    lo, hi = boot_auc(labels, scores if roc_auc_score(labels, scores) > 0.5 else -scores)
    
    auc_results.append({
        'feature': k,
        'auc': auc,
        'lo': lo,
        'hi': hi
    })

df_auc = pd.DataFrame(auc_results).sort_values('auc', ascending=False)
print(df_auc)

In [ ]:
# Pivot features for Nadler
feat_dict_for_fusion = {k: np.array([f[k] for f in all_feats]) for k in feat_keys}

best_a, best_lo, best_hi, best_s = best_nadler_on(
    feat_dict_for_fusion, 
    np.array(labels), 
    max_size=4, 
    min_size=3,
    normalize=True
)

In [ ]:
import matplotlib.pyplot as plt

def plot_trajectories(valid_statements, labels, n=5):
    plt.figure(figsize=(12, 5))
    
    grounded_idxs = [i for i, l in enumerate(labels) if l == 1][:n]
    ungrounded_idxs = [i for i, l in enumerate(labels) if l == 0][:n]
    
    for idx in grounded_idxs:
        plt.plot(valid_statements[idx]['ents'], color='blue', alpha=0.3, label='Grounded' if idx == grounded_idxs[0] else "")
    
    for idx in ungrounded_idxs:
        plt.plot(valid_statements[idx]['ents'], color='red', alpha=0.3, label='Ungrounded' if idx == ungrounded_idxs[0] else "")
        
    plt.title(f"H(n) Trajectories: Grounded (Blue) vs Ungrounded (Red)")
    plt.xlabel("Token index (local to statement)")
    plt.ylabel("Entropy H(n)")
    plt.legend()
    plt.show()

plot_trajectories(valid_statements, labels)

In [ ]:
import seaborn as sns
from scipy.stats import spearmanr

feat_matrix = np.array([ [f[k] for k in feat_keys] for f in all_feats])
corr, _ = spearmanr(feat_matrix)

plt.figure(figsize=(10, 8))
sns.heatmap(corr, xticklabels=feat_keys, yticklabels=feat_keys, annot=True, fmt=".2f", cmap='coolwarm')
plt.title("Feature Correlation (Spearman)")
plt.show()

In [ ]:
max_auc = df_auc['auc'].max()
fusion_auc = best_a

print("--- PILOT VERDICT ---")
print(f"Best Individual Feature AUC: {max_auc:.3f}")
print(f"Nadler Fusion AUC:          {fusion_auc:.3f}")

if max_auc > 0.60:
    print("PASS: Phase 10 proceeds to full scale.")
elif max_auc > 0.55:
    print("MARGINAL: Run FACTS Grounding before deciding.")
else:
    print("FAIL: Pivot to separate chapters (Plan A).")

In [ ]:
final_results = {
    'config': {
        'model': MODEL_ID,
        'task': TASK,
        'samples': N_SAMPLES
    },
    'auc_table': df_auc,
    'nadler': {
        'auc': best_a,
        'subset': best_s
    },
    'statement_metadata': valid_statements
}

with open(os.path.join(CACHE_DIR, f"{EXP_NAME}_results.pkl"), 'wb') as f:
    pickle.dump(final_results, f)
print("Results saved to Drive.")